# Residual RNN Training for STM32 Deployment

This notebook trains a compact RNN that predicts temperature better than the naive 24-hour-prior baseline by learning a residual correction on top of that baseline. It keeps the architecture TFLite-friendly and exports a fully quantized int8 model.

In [36]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [37]:
SEQ_LENGTH = 48          # 24 hours of 30-minute samples
HORIZON = 24             # next 12 hours at 30-minute resolution
RESAMPLE_RULE = '30min'
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

**Explanation:**
`SEQ_LENGTH = 48`:
- Each input example contains 48 time steps 
- Since data is resampled to 30-minute intervals, that corresponds to 48 * 30 minutes = 1,440 minutes = 24 hours

`HORIZON = 24`:
- The model predicts the next 24 time steps.
- Since data is resampled to 30-minute interval, that corresponds to 24 * 30 minutes = 720 minutes = 12 hours

`RESAMPLE_RULE = '30min'`:
- Tells Pandas to aggregate the raw sensor readings into one row every 30 minutes.
- Basically sensors capture data at slightly different times, so we aggregate them within a 30 minute window.

`BATCH_SIZE = 64`:
- When training the model, 64 examples are processed before updating the model weights.

`AUTOTUNE = tf.data.autotune`:
- Tells TensorFlow to automatically decide how aggressively it should prepare the next batches of data in the background while the model is training.

In [38]:
EXCLUDED_FILES = {
    'measurements_2000-01-01.csv',
    'power_measurements.csv',
}

ARTIFACT_DIR = Path('machine_learning/rnn/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_TFLITE_PATH = Path('machine_learning/rnn/residual_rnn_int8.tflite')

repo_root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((path for path in repo_root_candidates if (path / 'measurements').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root. Checked: {repo_root_candidates}')

DATA_DIR = REPO_ROOT / 'measurements'
print('Repo root:', REPO_ROOT.resolve())
print('Artifact dir:', ARTIFACT_DIR.resolve())
print('Tracked TFLite export path:', EXPORT_TFLITE_PATH.resolve())

Repo root: /Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting
Artifact dir: /Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/artifacts
Tracked TFLite export path: /Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/residual_rnn_int8.tflite


In [ ]:
csv_files = sorted(
    path for path in DATA_DIR.glob('*measurements*.csv')
    if path.name not in EXCLUDED_FILES
)

if not csv_files:
    raise FileNotFoundError(f'No measurement CSV files found in {DATA_DIR}')

frames = []
skipped_files = []
for csv_file in csv_files:
    if csv_file.stat().st_size == 0:
        skipped_files.append((csv_file.name, 'empty file'))
        continue
    try:
        df_temp = pd.read_csv(csv_file)
    except pd.errors.EmptyDataError:
        skipped_files.append((csv_file.name, 'empty data'))
        continue
    frames.append(df_temp)

if not frames:
    raise ValueError('No readable measurement CSVs were loaded.')

raw_df = pd.concat(frames, ignore_index=True)
raw_df = raw_df[raw_df['quantity'].isin(['temperature_c', 'humidity_pct', 'pressure_pa', 'lux_lx', 'is_raining'])].copy()
raw_df['timestamp_iso8601'] = pd.to_datetime(raw_df['timestamp_iso8601'], utc=True, errors='coerce')
raw_df['value'] = pd.to_numeric(raw_df['value'], errors='coerce')
raw_df = raw_df.dropna(subset=['timestamp_iso8601', 'value'])

raw_df.head(10)

Loaded files: 116
Skipped files: []
Raw rows: 3710079
Resampled rows: 9603


quantity,humidity_pct,is_raining,lux_lx,pressure_pa,temperature_c
timestamp_iso8601,,,,,
2025-09-04 22:00:00+00:00,63.341935,0.0,26.594607,100973.617845,28.172483
2025-09-04 22:30:00+00:00,69.947520,0.0,42.181484,100969.579486,28.347315
2025-09-04 23:00:00+00:00,75.398518,0.0,97.414887,100973.551907,30.579943
2025-09-04 23:30:00+00:00,76.887413,0.0,97.793978,101008.485453,30.319356
2025-09-05 00:00:00+00:00,79.289795,0.0,97.822113,101026.242403,29.734615


In [ ]:
wide_df = (
    raw_df
    .pivot_table(index='timestamp_iso8601', columns='quantity', values='value', aggfunc='mean')
    .sort_index()
)

resampled_df = wide_df.resample(RESAMPLE_RULE).mean()
resampled_df['is_raining'] = resampled_df['is_raining'].fillna(0.0).clip(0.0, 1.0)
numeric_cols = [col for col in resampled_df.columns if col != 'is_raining']
resampled_df[numeric_cols] = resampled_df[numeric_cols].interpolate(method='time', limit_direction='both')
resampled_df = resampled_df.dropna().copy()

print('Loaded files:', len(frames))
print('Skipped files:', skipped_files)
print('Raw rows:', len(raw_df))
print('Resampled rows:', len(resampled_df))
resampled_df.head()

In [40]:
feature_df = resampled_df.copy()
feature_df['temperature'] = feature_df['temperature_c']
feature_df['humidity'] = feature_df['humidity_pct']
feature_df['pressure'] = feature_df['pressure_pa']
feature_df['lux'] = feature_df['lux_lx']

hours = feature_df.index.hour + (feature_df.index.minute / 60.0)
feature_df['sin_hour'] = np.sin(2.0 * np.pi * hours / 24.0)
feature_df['cos_hour'] = np.cos(2.0 * np.pi * hours / 24.0)

feature_df['temp_delta_24h'] = feature_df['temperature'] - feature_df['temperature'].shift(48)
feature_df['pressure_delta_24h'] = feature_df['pressure'] - feature_df['pressure'].shift(48)
feature_df['temp_mean_24h'] = feature_df['temperature'].rolling(48).mean()

feature_df = feature_df.dropna().copy()

feature_cols = [
    'temperature',
    'humidity',
    'pressure',
    'lux',
    'temp_delta_24h',
    'pressure_delta_24h',
    'temp_mean_24h',
    'sin_hour',
    'cos_hour',
]

target_col = 'temperature_c'
temp_feature_index = feature_cols.index('temperature')
feature_df[feature_cols].head()

quantity,temperature,humidity,pressure,lux,temp_delta_24h,pressure_delta_24h,temp_mean_24h,sin_hour,cos_hour
timestamp_iso8601,,,,,,,,,
2025-09-05 22:00:00+00:00,30.032821,78.994840,101078.715203,0.563733,1.860338,105.097358,29.887603,-0.500000,0.866025
2025-09-05 22:30:00+00:00,29.732960,80.726784,101104.470644,0.000000,1.385644,134.891158,29.916471,-0.382683,0.923880
2025-09-05 23:00:00+00:00,29.993142,80.424112,101109.030558,82.750078,-0.586801,135.478651,29.904246,-0.258819,0.965926
2025-09-05 23:30:00+00:00,30.352780,78.597592,101141.617850,90.148348,0.033424,133.132397,29.904942,-0.130526,0.991445
2025-09-06 00:00:00+00:00,30.348785,77.913760,101169.385776,89.890557,0.614170,143.143373,29.917737,0.000000,1.000000


In [41]:
n_rows = len(feature_df)
train_row_end = int(n_rows * 0.70)
valid_row_end = int(n_rows * 0.85)

train_rows = feature_df.iloc[:train_row_end].copy()
valid_rows = feature_df.iloc[train_row_end:valid_row_end].copy()
test_rows = feature_df.iloc[valid_row_end:].copy()

scale_cols = [
    'humidity',
    'pressure',
    'lux',
    'temp_delta_24h',
    'pressure_delta_24h',
    'temp_mean_24h',
]

scaler = StandardScaler()
scaler.fit(train_rows[scale_cols])

scaled_feature_df = feature_df.copy()
scaled_feature_df.loc[:, scale_cols] = scaler.transform(feature_df[scale_cols])

train_cut_timestamp = feature_df.index[train_row_end]
valid_cut_timestamp = feature_df.index[valid_row_end]

print('Train rows:', len(train_rows))
print('Valid rows:', len(valid_rows))
print('Test rows:', len(test_rows))
print('Train cut:', train_cut_timestamp)
print('Valid cut:', valid_cut_timestamp)

Train rows: 6688
Valid rows: 1433
Test rows: 1434
Train cut: 2026-01-23 06:00:00+00:00
Valid cut: 2026-02-22 02:30:00+00:00


In [42]:
def build_supervised_windows(df, feature_columns, target_column, seq_length, horizon):
    feature_matrix = df[feature_columns].to_numpy(dtype=np.float32)
    target_vector = df[target_column].to_numpy(dtype=np.float32)
    timestamps = df.index.to_numpy()

    x_list = []
    y_list = []
    baseline_list = []
    target_timestamps = []

    max_start = len(df) - seq_length - horizon + 1
    for start in range(max_start):
        stop = start + seq_length
        horizon_stop = stop + horizon

        input_window = feature_matrix[start:stop]
        future_target = target_vector[stop:horizon_stop]

        # For each future step, use the value from the same half-hour one day earlier.
        previous_day_baseline = input_window[:horizon, temp_feature_index]

        x_list.append(input_window)
        y_list.append(future_target)
        baseline_list.append(previous_day_baseline)
        target_timestamps.append(timestamps[stop])

    return (
        np.asarray(x_list, dtype=np.float32),
        np.asarray(y_list, dtype=np.float32),
        np.asarray(baseline_list, dtype=np.float32),
        np.asarray(target_timestamps),
    )


X_all, y_all, baseline_all, sample_timestamps = build_supervised_windows(
    scaled_feature_df,
    feature_cols,
    target_col,
    SEQ_LENGTH,
    HORIZON,
)

sample_timestamps = pd.to_datetime(sample_timestamps, utc=True)
train_cut_ts = pd.Timestamp(train_cut_timestamp)
valid_cut_ts = pd.Timestamp(valid_cut_timestamp)
if train_cut_ts.tzinfo is None:
    train_cut_ts = train_cut_ts.tz_localize('UTC')
else:
    train_cut_ts = train_cut_ts.tz_convert('UTC')
if valid_cut_ts.tzinfo is None:
    valid_cut_ts = valid_cut_ts.tz_localize('UTC')
else:
    valid_cut_ts = valid_cut_ts.tz_convert('UTC')

train_mask = sample_timestamps < train_cut_ts
valid_mask = (sample_timestamps >= train_cut_ts) & (sample_timestamps < valid_cut_ts)
test_mask = sample_timestamps >= valid_cut_ts

X_train, y_train, baseline_train = X_all[train_mask], y_all[train_mask], baseline_all[train_mask]
X_valid, y_valid, baseline_valid = X_all[valid_mask], y_all[valid_mask], baseline_all[valid_mask]
X_test, y_test, baseline_test = X_all[test_mask], y_all[test_mask], baseline_all[test_mask]

print('Train windows:', X_train.shape, y_train.shape)
print('Valid windows:', X_valid.shape, y_valid.shape)
print('Test windows:', X_test.shape, y_test.shape)

baseline_valid_mae = float(np.mean(np.abs(y_valid - baseline_valid)))
baseline_test_mae = float(np.mean(np.abs(y_test - baseline_test)))
print(f'Naive day-ago baseline valid MAE: {baseline_valid_mae:.4f} C')
print(f'Naive day-ago baseline test MAE:  {baseline_test_mae:.4f} C')

Train windows: (6640, 48, 9) (6640, 24)
Valid windows: (1433, 48, 9) (1433, 24)
Test windows: (1411, 48, 9) (1411, 24)
Naive day-ago baseline valid MAE: 0.0543 C
Naive day-ago baseline test MAE:  1.0018 C


In [43]:
def make_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(min(len(X), 8192), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(X_train, y_train, training=True)
valid_ds = make_dataset(X_valid, y_valid, training=False)
test_ds = make_dataset(X_test, y_test, training=False)

horizon_weights = tf.constant(np.linspace(1.0, 1.35, HORIZON), dtype=tf.float32)

def weighted_huber(y_true, y_pred):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    delta = 1.5
    quadratic = tf.minimum(abs_error, delta)
    linear = abs_error - quadratic
    huber = 0.5 * tf.square(quadratic) + delta * linear
    weighted = huber * horizon_weights
    return tf.reduce_mean(weighted, axis=-1)


def build_residual_rnn(config):
    inputs = tf.keras.Input(shape=(SEQ_LENGTH, len(feature_cols)), name='sensor_history')
    baseline = tf.keras.layers.Lambda(
        lambda x: x[:, :HORIZON, temp_feature_index],
        name='previous_day_baseline',
    )(inputs)

    x = tf.keras.layers.SimpleRNN(
        config['rnn_units_1'],
        return_sequences=True,
        unroll=True,
        activation='tanh',
        name='rnn_1',
    )(inputs)
    x = tf.keras.layers.Dropout(config['dropout'], name='dropout_1')(x)
    x = tf.keras.layers.SimpleRNN(
        config['rnn_units_2'],
        unroll=True,
        activation='tanh',
        name='rnn_2',
    )(x)
    x = tf.keras.layers.Dense(config['dense_units'], activation='relu', name='dense_1')(x)
    residual = tf.keras.layers.Dense(HORIZON, name='residual_delta_c')(x)
    outputs = tf.keras.layers.Add(name='forecast_c')([baseline, residual])

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='residual_rnn_forecaster')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config['learning_rate']),
        loss=weighted_huber,
        metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')],
    )
    return model


search_configs = [
    {'name': 'cfg_small',  'rnn_units_1': 24, 'rnn_units_2': 16, 'dense_units': 24, 'dropout': 0.05, 'learning_rate': 2e-3},
    {'name': 'cfg_mid',    'rnn_units_1': 32, 'rnn_units_2': 24, 'dense_units': 32, 'dropout': 0.10, 'learning_rate': 1e-3},
    {'name': 'cfg_wide',   'rnn_units_1': 40, 'rnn_units_2': 24, 'dense_units': 32, 'dropout': 0.10, 'learning_rate': 8e-4},
    {'name': 'cfg_deep',   'rnn_units_1': 32, 'rnn_units_2': 32, 'dense_units': 40, 'dropout': 0.15, 'learning_rate': 8e-4},
    {'name': 'cfg_edge',   'rnn_units_1': 48, 'rnn_units_2': 24, 'dense_units': 32, 'dropout': 0.05, 'learning_rate': 7e-4},
]

def count_params_for_config(config):
    return build_residual_rnn(config).count_params()

[(cfg['name'], count_params_for_config(cfg)) for cfg in search_configs]

[('cfg_small', 2480),
 ('cfg_mid', 4304),
 ('cfg_wide', 5152),
 ('cfg_deep', 5728),
 ('cfg_edge', 6128)]

In [44]:
training_results = []
best_model = None
best_history = None
best_config = None
best_valid_mae = np.inf

for config in search_configs:
    print(f"\n=== Training {config['name']} ===")
    model = build_residual_rnn(config)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_mae', patience=8, mode='min', restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=3, min_lr=1e-5, mode='min'),
    ]

    history = model.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=50,
        verbose=1,
        callbacks=callbacks,
    )

    valid_metrics = model.evaluate(valid_ds, verbose=0, return_dict=True)
    params = model.count_params()
    training_results.append({
        'name': config['name'],
        'params': params,
        'val_loss': float(valid_metrics['loss']),
        'val_mae': float(valid_metrics['mae']),
    })
    print(f"Validation MAE for {config['name']}: {valid_metrics['mae']:.4f} C | params={params:,}")

    if valid_metrics['mae'] < best_valid_mae:
        best_valid_mae = float(valid_metrics['mae'])
        best_model = model
        best_history = history.history
        best_config = config

results_df = pd.DataFrame(training_results).sort_values(['val_mae', 'params']).reset_index(drop=True)
print('\nSearch results:')
display(results_df)
print('Best config:', best_config)


=== Training cfg_small ===
Epoch 1/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - loss: 0.9255 - mae: 0.9146 - val_loss: 0.0049 - val_mae: 0.0853 - learning_rate: 0.0020
Epoch 2/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.8955 - mae: 0.8825 - val_loss: 0.0198 - val_mae: 0.1650 - learning_rate: 0.0020
Epoch 3/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.7997 - mae: 0.8127 - val_loss: 0.0160 - val_mae: 0.1490 - learning_rate: 0.0020
Epoch 4/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.7769 - mae: 0.7890 - val_loss: 0.0138 - val_mae: 0.1405 - learning_rate: 0.0020
Epoch 5/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.7662 - mae: 0.7710 - val_loss: 0.0073 - val_mae: 0.1004 - learning_rate: 0.0010
Epoch 6/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.7607 - mae: 0.7652 - val_loss: 0.0059 - val_mae: 0.0882 - learning_rate: 0.0010
Epoch 7/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.7558 - mae: 0.7636 - val_loss: 0.0037 - val_mae: 0

,name,params,val_loss,val_mae
0,cfg_small,2480,0.001721,0.043251
1,cfg_deep,5728,0.004027,0.065913
2,cfg_mid,4304,0.003311,0.070946
3,cfg_wide,5152,0.006603,0.086788
4,cfg_edge,6128,0.007703,0.087216


Best config: {'name': 'cfg_small', 'rnn_units_1': 24, 'rnn_units_2': 16, 'dense_units': 24, 'dropout': 0.05, 'learning_rate': 0.002}


In [45]:
final_metrics = {
    'baseline_valid_mae_c': baseline_valid_mae,
    'baseline_test_mae_c': baseline_test_mae,
}

valid_eval = best_model.evaluate(valid_ds, verbose=0, return_dict=True)
test_eval = best_model.evaluate(test_ds, verbose=0, return_dict=True)
final_metrics['model_valid_mae_c'] = float(valid_eval['mae'])
final_metrics['model_test_mae_c'] = float(test_eval['mae'])
final_metrics['test_mae_gain_c'] = baseline_test_mae - float(test_eval['mae'])
final_metrics['test_skill_vs_baseline_pct'] = 100.0 * final_metrics['test_mae_gain_c'] / max(baseline_test_mae, 1e-6)

y_valid_pred = best_model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0)
y_test_pred = best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

per_horizon_test_mae = np.mean(np.abs(y_test - y_test_pred), axis=0)
per_horizon_baseline_mae = np.mean(np.abs(y_test - baseline_test), axis=0)

metrics_df = pd.DataFrame([final_metrics])
display(metrics_df)

comparison_df = pd.DataFrame({
    'step_ahead': np.arange(1, HORIZON + 1),
    'baseline_mae_c': per_horizon_baseline_mae,
    'model_mae_c': per_horizon_test_mae,
    'gain_c': per_horizon_baseline_mae - per_horizon_test_mae,
})
display(comparison_df)

best_model_path = ARTIFACT_DIR / 'best_residual_rnn.keras'
best_model.save(best_model_path)
print('Saved best Keras model to:', best_model_path.resolve())

if final_metrics['model_test_mae_c'] >= baseline_test_mae:
    print('Model is not yet beating the day-ago baseline on test. Expand the search grid or train longer before deployment.')
else:
    print('Model beats the day-ago baseline on test and is ready for quantization.')

,baseline_valid_mae_c,baseline_test_mae_c,model_valid_mae_c,model_test_mae_c,test_mae_gain_c,test_skill_vs_baseline_pct
0,0.054303,1.001787,0.043251,0.913819,0.087968,8.781126


,step_ahead,baseline_mae_c,model_mae_c,gain_c
0,1,0.994492,0.476560,0.517932
1,2,0.994651,0.557047,0.437604
2,3,0.995048,0.635717,0.359332
3,4,0.995939,0.709606,0.286333
4,5,0.996559,0.767100,0.229459
5,6,0.996888,0.828433,0.168455
6,7,0.997085,0.883763,0.113323
7,8,0.997201,0.925587,0.071614
8,9,0.997224,0.949638,0.047586
9,10,0.997456,0.985653,0.011803


Saved best Keras model to: /Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/artifacts/best_residual_rnn.keras
Model beats the day-ago baseline on test and is ready for quantization.


In [46]:
best_model.summary()

def estimate_tflite_sram_kb(tflite_bytes):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    tensor_details = interpreter.get_tensor_details()

    def tensor_ram_bytes(detail):
        dtype = np.dtype(detail['dtype'])
        shape = detail.get('shape_signature', detail.get('shape'))
        shape = [1 if dim < 0 else dim for dim in shape]
        return int(np.prod(shape)) * dtype.itemsize

    sram_bytes = sum(
        tensor_ram_bytes(detail)
        for detail in tensor_details
        if detail.get('buffer', 0) == 0 or detail.get('is_variable', False)
    )
    return sram_bytes / 1024.0

def representative_dataset():
    for batch_x, _ in train_ds.unbatch().batch(1).take(400):
        yield [tf.cast(batch_x, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_bytes = converter.convert()
tflite_path = ARTIFACT_DIR / 'residual_rnn_int8.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_int8_bytes)
with open(EXPORT_TFLITE_PATH, 'wb') as f:
    f.write(tflite_int8_bytes)

size_bytes = tflite_path.stat().st_size
size_kb = size_bytes / 1024.0
estimated_sram_kb = estimate_tflite_sram_kb(tflite_int8_bytes)

print(f'TFLite int8 artifact path: {tflite_path.resolve()}')
print(f'TFLite int8 tracked path: {EXPORT_TFLITE_PATH.resolve()}')
print(f'TFLite int8 size: {size_bytes} bytes ({size_kb:.2f} KB)')
print(f'Estimated tensor SRAM: {estimated_sram_kb:.2f} KB')

Model: "residual_rnn_forecaster"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor_history      │ (None, 48, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_1 (SimpleRNN)   │ (None, 48, 24)    │        816 │ sensor_history[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 48, 24)    │          0 │ rnn_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_2 (SimpleRNN)   │ (None, 16)        │        656 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 24)        │        408 │ rnn_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ previous_day_basel… │ (None, 24)        │          0 │ sensor_history[0… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_delta_c    │ (None, 24)        │        600 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forecast_c (Add)    │ (None, 24)        │          0 │ previous_day_bas… │
│                     │                   │            │ residual_delta_c… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,442 (29.07 KB)

 Trainable params: 2,480 (9.69 KB)

 Non-trainable params: 0 (0.00 B)

...

INFO:tensorflow:Assets written to: /var/folders/x7/m1y0b5hs6lvdcp8n01m1fnp00000gn/T/tmpliz9l5wi/assets


Saved artifact at '/var/folders/x7/m1y0b5hs6lvdcp8n01m1fnp00000gn/T/tmpliz9l5wi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 48, 9), dtype=tf.float32, name='sensor_history')
Output Type:
  TensorSpec(shape=(None, 24), dtype=tf.float32, name=None)
Captures:
  5065209136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5089899648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4760892192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5089898768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5060402992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5060164272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5060400704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4978694048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5060405456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4978694928: TensorSpec(shape=(), dtype=tf.resource, name=None)


/Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774523571.316101 17038601 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1774523571.316848 17038601 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1774523571.317709 17038601 reader.cc:83] Reading SavedModel from: /var/folders/x7/m1y0b5hs6lvdcp8n01m1fnp00000gn/T/tmpliz9l5wi
I0000 00:00:1774523571.321301 17038601 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1774523571.321321 17038601 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/x7/m1y0b5hs6lvdcp8n01m1fnp00000gn/T/tmpliz9l5wi
I0000 00:00:1774523571.364002 17038601 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1774523571.497941 17038601 loader.cc:220] Running initiali

TFLite int8 artifact path: /Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/artifacts/residual_rnn_int8.tflite
TFLite int8 tracked path: /Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/residual_rnn_int8.tflite
TFLite int8 size: 354608 bytes (346.30 KB)
Estimated tensor SRAM: 23.53 KB


/Users/tevinachong/Documents/TinyML-Home-Weather-Forecasting/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [47]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print('Input details:', input_details)
print('Output details:', output_details)

input_scale, input_zero_point = input_details['quantization']
output_scale, output_zero_point = output_details['quantization']

def quantize_input(x):
    return np.clip(np.round(x / input_scale + input_zero_point), -128, 127).astype(np.int8)

def dequantize_output(x):
    return (x.astype(np.float32) - output_zero_point) * output_scale

tflite_preds = []
for i in range(len(X_test)):
    sample_x = X_test[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details['index'], quantize_input(sample_x))
    interpreter.invoke()
    raw_output = interpreter.get_tensor(output_details['index'])
    tflite_preds.append(dequantize_output(raw_output)[0])

tflite_preds = np.asarray(tflite_preds, dtype=np.float32)
tflite_test_mae = float(np.mean(np.abs(y_test - tflite_preds)))
print(f'TFLite int8 test MAE: {tflite_test_mae:.4f} C')
print(f'Test baseline MAE:   {baseline_test_mae:.4f} C')
print(f'TFLite gain vs baseline: {baseline_test_mae - tflite_test_mae:.4f} C')

preview_index = 0
preview_df = pd.DataFrame({
    'actual_c': y_test[preview_index],
    'baseline_c': baseline_test[preview_index],
    'keras_pred_c': y_test_pred[preview_index],
    'tflite_pred_c': tflite_preds[preview_index],
})

preview_df["baseline_diff"] = np.abs(preview_df["baseline_c"] - preview_df["actual_c"])
preview_df["keras_diff"] = np.abs(preview_df["keras_pred_c"] - preview_df["actual_c"])
preview_df["tflite_diff"] = np.abs(preview_df["tflite_pred_c"] - preview_df["actual_c"])
display(preview_df)


Input details: {'name': 'serving_default_sensor_history:0', 'index': 0, 'shape': array([ 1, 48,  9], dtype=int32), 'shape_signature': array([-1, 48,  9], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.19839172065258026, -92), 'quantization_parameters': {'scales': array([0.19839172], dtype=float32), 'zero_points': array([-92], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}
Output details: {'name': 'StatefulPartitionedCall_1:0', 'index': 657, 'shape': array([ 1, 24], dtype=int32), 'shape_signature': array([-1, 24], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.16967834532260895, -128), 'quantization_parameters': {'scales': array([0.16967835], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}
TFLite int8 test MAE: 0.9294 C
Test baseline MAE:   1.0018 C
TFLite gain vs baseline: 0.0724 C


,actual_c,baseline_c,keras_pred_c,tflite_pred_c,baseline_diff,keras_diff,tflite_diff
0,28.533255,28.478952,28.516842,28.505962,0.054302,0.016413,0.027292
1,28.534386,28.480083,28.537165,28.505962,0.054302,0.002779,0.028423
2,28.535517,28.481215,28.539055,28.675640,0.054302,0.003538,0.140123
3,28.536650,28.482346,28.546547,28.675640,0.054304,0.009897,0.138990
4,28.537781,28.483477,28.538881,28.675640,0.054304,0.001101,0.137859
5,28.538912,28.484610,28.519543,28.675640,0.054302,0.019369,0.136728
6,28.540043,28.485741,28.537472,28.675640,0.054302,0.002571,0.135597
7,28.541174,28.486872,28.544149,28.675640,0.054302,0.002975,0.134466
8,28.542305,28.488003,28.563271,28.675640,0.054302,0.020966,0.133335
9,28.543436,28.489134,28.572731,28.675640,0.054302,0.029295,0.132204


In [48]:
print(f"Baseline MAE: {preview_df['baseline_diff'].mean():.4f} C")
print(f"Full Model MAE: {preview_df['keras_diff'].mean():.4f} C")
print(f"TFLite MAE: {preview_df['tflite_diff'].mean():.4f} C")

Baseline MAE: 0.0543 C
Full Model MAE: 0.0232 C
TFLite MAE: 0.1117 C
